# Ablation Study (leave-one-out)

Isolates the contribution of each ICDN component by disabling one module at a
time and re-running a small Optuna search per variant. Two evaluation axes,
matching the referee's request: predictive performance (R2 / MAE / RMSE) and
elasticity stability + plausibility (elast_score and its cross-fold std).

Variants (leave-one-out): full, no_smooth, no_elast, no_attention, no_cross, no_splines.
No seed loop, no bootstrap: stability is estimated across temporal folds only.
No changes to src/ are required:
  - cross-price  -> existing use_cross=False flag
  - attention    -> uniform-weight monkeypatch of the neighbor selector
  - splines      -> zero + freeze of the spline heads

# Imports

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import json
import types
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import optuna
from pathlib import Path

# -- Personal Libraries
from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, ProductTokenBuilder
from src.utils import TemporalSplitter
from src.nn.heads.neighbor_selector import SparseNeighborSelector 
from src.nn.loss.elasticity_mask import elasticity_entry_mask

/home/thebigmonster/Github/nn-elasticity/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Settings 

In [ ]:
# initial seed
BASE_SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Data ──────────────────────────────────────────────────────────
N_UPCS = 5
SMOOTH_WINDOW = 8
BETA_EDA = -2
K_NEIGHBORS = 5

# ── Nested temporal (mismo protocolo que ICDN vs MLP) ─────────────
PROTOCOL = "nested_temporal"
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
TUNE_SEEDS = [11, 29, 42]
EVAL_SEEDS = [11, 29, 42, 77, 123]
MIN_TRAIN_FRAC = 0.50
TRAIN_FRAC = 0.8
N_TRIALS_NESTED = 20
N_TRIALS_HOLDOUT = 20

# ── Training (tune; eval puede usar más epochs en nn_final_evaluation) ──
N_EPOCHS_P0 = 250
N_EPOCHS_P1 = 300
PATIENCE    = 20
ES_PATIENCE = 40

D_STORE = 16
D_BRAND = 8
D_STYLE = 8

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NESTED_ABL_DIR = RESULTS_DIR / "nested" / "ablation"
NESTED_ABL_DIR.mkdir(parents=True, exist_ok=True)

CKPT_DIR = Path("../results/checkpoints/ablation_nested")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Seeds

In [4]:
# Function to set all seeds and make the results reproducible
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

# Code - Loader

In [ ]:
# Load the dataset
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()
print(f"Dataset shape: {df.shape}")

# Encode the categorical variables
encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id", sort=True)
_, brand_cats = encoder.factorize(df, "brand_family_norm",  sort=True)
_, style_cats = encoder.factorize(df, "style_segment_norm", sort=True)

n_stores = len(store_cats)
n_weeks  = len(week_cats)
n_brands = len(brand_cats)
n_styles = len(style_cats)
print(f"Stores: {n_stores}  |  Weeks: {n_weeks}  |  Brands: {n_brands}  |  Styles: {n_styles}")

# Map brand/style numeric codes to 1,2,... (0 reserved for unknown/missing)
brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"]  = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

# Build the multi-product (wide-format) dataset
mp_builder = MultiProductBuilder()
sorted_weeks = sorted(df["week_id"].unique())
n_sample = max(1, int(len(sorted_weeks) * MIN_TRAIN_FRAC))
sample_weeks = sorted_weeks[:n_sample]
mp_builder.fit_panel(df[df["week_id"].isin(sample_weeks)], n_upcs=N_UPCS)
n_upcs = mp_builder.n

print(f"UPCs selected: {n_upcs}")
print(f"Top UPCs: {mp_builder.selected_upcs[:N_UPCS]}")

Dataset shape: (463722, 44)
Stores: 70  |  Weeks: 302  |  Brands: 54  |  Styles: 13
Full wide shape: (19808, 171)
UPCs selected: 5
Top UPCs: [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]


# Neighbor meta

In [6]:
# Static metadata per UPC position — used by neighbor-aware attention in the model
upc_meta = (
    df.groupby("upc_code")[["category_code", "brand_family_norm",
                             "style_segment_norm", "liters_per_upc"]]
    .first()
    .loc[mp_builder.selected_upcs]
)

cat_codes, _ = pd.factorize(upc_meta["category_code"], sort=True)

neighbor_meta = {
    "category": torch.tensor(cat_codes, dtype=torch.long, device=device),
    "brand":    torch.tensor(upc_meta["brand_family_norm"].values,   dtype=torch.long,    device=device),
    "style":    torch.tensor(upc_meta["style_segment_norm"].values,  dtype=torch.long,    device=device),
    "liters":   torch.tensor(upc_meta["liters_per_upc"].values,      dtype=torch.float32, device=device),
}
print("neighbor_meta built")

neighbor_meta built


# Temporal Folds

In [ ]:
splitter = TemporalSplitter(week_col="week_id")

nested_plans = splitter.nested_expanding_splits(
    df=df,
    n_outer=N_OUTER_FOLDS,
    n_inner=N_INNER_FOLDS,
    min_train_frac=MIN_TRAIN_FRAC,
)

def materialize(plan_or_pair):
    if isinstance(plan_or_pair, dict):
        tr_w, va_w = mp_builder.make_fold_frames(
            plan_or_pair["outer_train"], plan_or_pair["outer_val"]
        )
        plan_or_pair["outer_train"], plan_or_pair["outer_val"] = tr_w, va_w
        plan_or_pair["inner_splits"] = [
            mp_builder.make_fold_frames(tr, va)
            for tr, va in plan_or_pair["inner_splits"]
        ]
        return plan_or_pair
    tr, va = plan_or_pair
    return mp_builder.make_fold_frames(tr, va)

nested_plans = [materialize(p) for p in nested_plans]

train_final_long, val_final_long = splitter.single_split(df, train_frac=TRAIN_FRAC)
holdout_inner_long = splitter.expanding_splits(
    train_final_long, n_folds=N_INNER_FOLDS, min_train_frac=MIN_TRAIN_FRAC,
)
holdout_weeks = set(val_final_long["week_id"].unique())
for _, inner_val in holdout_inner_long:
    leak = set(inner_val["week_id"].unique()) & holdout_weeks
    if leak:
        raise RuntimeError(f"Holdout leak: {sorted(leak)[:10]}")

holdout_inner = [mp_builder.make_fold_frames(tr, va) for tr, va in holdout_inner_long]
train_final, val_final = mp_builder.make_fold_frames(train_final_long, val_final_long)

print(f"outers={len(nested_plans)} inners/outer={N_INNER_FOLDS} holdout_inner={len(holdout_inner)}")
for p in nested_plans:
    print(
        f"  outer {p['outer_id']}: "
        f"train_weeks={p['outer_train']['week_id'].nunique()} "
        f"val_weeks={p['outer_val']['week_id'].nunique()}"
    )

# Fold-Frame + loader Helpers

In [ ]:
# Global store/week code maps (must stay consistent across folds)
store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}


# Encodes store/week codes, sorts, and smooths log liters for phase 0.
def build_fold_frames(train_wide, val_wide, smooth_window: int):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    def _smooth(df_w):
        out = df_w.sort_values(["store_code", "week_id"]).copy()
        for i in range(n_upcs):
            y = out[f"log_liters_{i}"].where(out[f"obs_mask_{i}"].eq(1))
            out[f"log_liters_{i}"] = (
                y.groupby(out["store_code"])
                 .transform(lambda s: s.rolling(window=smooth_window, min_periods=1).mean())
                 .fillna(0.0)
            )
        return out

    train_wide_s = _smooth(train_wide)
    val_wide_s   = _smooth(val_wide)
    return train_wide, val_wide, train_wide_s, val_wide_s


# Builds the phase-0 (smoothed) and phase-1 (raw) train/val DataLoaders.
def build_loaders(train_wide, val_wide, train_wide_s, val_wide_s, batch_size: int):
    loader_factory = DataLoaderFactory(
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )

    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs)
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs)
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs)
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs)

    train_loader_p0 = loader_factory.create_train_loader(train_ds_p0, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader_p0   = loader_factory.create_eval_loader(val_ds_p0,    batch_size=batch_size, shuffle=False)
    train_loader    = loader_factory.create_train_loader(train_ds,    batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader      = loader_factory.create_eval_loader(val_ds,       batch_size=batch_size, shuffle=False)
    return train_loader_p0, val_loader_p0, train_loader, val_loader

# Freeze - Init Helpers

In [ ]:
def zero_and_freeze_nonlinear(model):
    ph = model.head.param_head
    for attr in ("head_w", "head_w_cross", "head_cross"):
        if not hasattr(ph, attr):
            continue
        layer = getattr(ph, attr)
        with torch.no_grad():
            layer.weight.zero_()
            if layer.bias is not None:
                layer.bias.zero_()
        layer.weight.requires_grad_(False)
        if layer.bias is not None:
            layer.bias.requires_grad_(False)

def unfreeze_nonlinear(model):
    # Unfreeze all spline and bilinear heads for phase 1.
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        if hasattr(model.head.param_head, attr):
            head = getattr(model.head.param_head, attr)
            head.weight.requires_grad_(True)
            head.bias.requires_grad_(True)

# Initialize the beta prior so the INITIAL own-price slope equals beta_target
# (~ -2 from EDA) regardless of whether the negative-sign transform is active.
# NOTE: this is only an initialization (a soft nudge), not a constraint: with
# enforce_negative_beta=False the model is free to move beta to any sign.
def init_beta_prior(model, beta_target, enforce_negative_beta=True):
    with torch.no_grad():
        model.head.param_head.head_beta.weight.zero_()
        if enforce_negative_beta:
            # beta = -softplus(beta_raw): invert softplus to recover beta_target.
            beta_raw_init = torch.log(
                torch.exp(torch.tensor(-float(beta_target), dtype=torch.float32)) - 1.0
            )
        else:
            # beta = beta_raw directly: set the bias to beta_target as-is.
            beta_raw_init = torch.tensor(float(beta_target), dtype=torch.float32)
        model.head.param_head.head_beta.bias.fill_(beta_raw_init)

print("Helpers defined")

Helpers defined


# Ablation

In [ ]:
# ── Ablation switch 2: uniform attention (attention OFF) ───────────
# Keep a handle to the original run so both the online and frozen-graph paths
# are reused; we only overwrite the weights.
_ORIG_SELECTOR_RUN = SparseNeighborSelector.run

def _uniform_selector_run(self, h, category, brand, style, liters, availability):
    """Same neighbor graph as the learned selector, but replace softmax
    edge weights by uniform weights over *available* neighbors. Isolates
    *learned attention* from *having cross-price neighbors at all*."""
    pairs, _ = _ORIG_SELECTOR_RUN(
        self, h, category, brand, style, liters, availability,
    )
    B, n, _ = h.shape
    if pairs.numel() == 0:
        return pairs, h.new_empty(B, 0)

    k_eff = pairs.shape[1] // n
    available_j = availability[:, pairs[1]].view(B, n, k_eff).to(h.dtype)
    # Si ningún vecino está disponible, sum=0 → clamp_min(1) deja pesos 0
    # (mismo contrato que SparseNeighborSelector._softmax_available).
    weights = available_j / available_j.sum(dim=-1, keepdim=True).clamp_min(1.0)
    return pairs, weights.reshape(B, -1)

def disable_attention(model):
    """Bind the uniform-weight run() onto the selector instance, preserving the
    learned q/k projections that decide the graph structure."""
    sel = model.head.neighbor_selector
    if sel is not None:
        sel.run = types.MethodType(_uniform_selector_run, sel)

print("Ablation switches defined")

Ablation switches defined


# Metrics Helpers

In [ ]:
# Hidden options for the encoder (Optuna categorical)
HIDDEN_OPTIONS = {
    "64_32":        (64, 32),
    "128_64":       (128, 64),
    "192_96":       (192, 96),
    "256_128":      (256, 128),
    "256_128_64":   (256, 128, 64),
}

# Global predictive metrics on observed entries only.
def compute_global_metrics(model, val_loader, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true   = batch["demands"]
            obs_mask = batch["obs_mask"]
            y_hat, _, _ = model(batch, return_parts=True, neighbor_meta=neighbor_meta)

            mask = obs_mask.bool()
            all_true.append(y_true[mask].cpu())
            all_pred.append(y_hat[mask].cpu())

    y_true_all = torch.cat(all_true).float()
    y_pred_all = torch.cat(all_pred).float()

    err  = y_true_all - y_pred_all
    mae  = float(err.abs().mean())
    rmse = float(torch.sqrt((err ** 2).mean()))

    ss_res = float((err ** 2).sum())
    ss_tot = float(((y_true_all - y_true_all.mean()) ** 2).sum())
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {"mae_val": mae, "rmse_val": rmse, "r2_val": r2}


# Elasticity plausibility score in [0, 1]; higher is better.
# Own-price target range [-5, 0], cross-price target range [-1, 1].
def compute_elasticity_score(model, val_loader, device,
                             own_min=-5.0, own_max=0.0,
                             cross_min=-1.0, cross_max=1.0):
    model.eval()
    all_own, all_cross = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            obs_mask = batch["obs_mask"].bool()
            _, eps_hat, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)

            E = aux["E"]
            n = E.shape[1]
            M = elasticity_entry_mask(
                batch["obs_mask"],
                pairs=aux["pairs"],
                availability=batch["availability"],
                price_observed=batch["price_observed"],
            )
            own_m = torch.diagonal(M, dim1=1, dim2=2)
            all_own.append(torch.diagonal(E, dim1=1, dim2=2)[own_m].cpu())
            eye = torch.eye(n, dtype=torch.bool, device=E.device)
            all_cross.append(E[M & ~eye.unsqueeze(0)].cpu())

    own   = torch.cat(all_own).numpy()
    cross = torch.cat(all_cross).numpy() if all_cross else np.array([])

    # ── Own score: fraction in range, penalised by deviation from the EDA prior ──
    own_in_range  = float(((own >= own_min) & (own <= own_max)).mean())
    median_own    = float(np.median(own))
    deviation     = max(0.0, abs(median_own - BETA_EDA) - 0.3)
    prior_penalty = min(deviation / abs(BETA_EDA), 1.0)
    own_score     = own_in_range * (1.0 - prior_penalty)

    # ── Cross score: fraction in range ──
    if len(cross) > 0:
        cross_in_range = float(((cross >= cross_min) & (cross <= cross_max)).mean())
        median_cross   = float(np.median(cross))
    else:
        cross_in_range = 1.0
        median_cross   = float("nan")

    score = 0.7 * own_score + 0.3 * cross_in_range

    return {
        "elast_score":             float(score),
        "own_score":               float(own_score),
        "own_in_range":            float(own_in_range),
        "own_elasticity_median":   median_own,
        "cross_in_range":          float(cross_in_range),
        "cross_elasticity_median": median_cross,
    }

# Descriptive, target-free plausibility report. Unlike compute_elasticity_score
# (whose ranges coincide with the training penalty and the selection criterion),
# these are raw distributional facts. The most honest signal is own_frac_negative
# under the unconstrained variant: sign that survives WITHOUT being imposed.
def compute_plausibility_report(model, val_loader, device):
    model.eval()
    all_own, all_cross = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            _, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
            E = aux["E"]
            n = E.shape[1]
            M = elasticity_entry_mask(
                batch["obs_mask"],
                pairs=aux["pairs"],
                availability=batch["availability"],
                price_observed=batch["price_observed"],
            )
            own_m = torch.diagonal(M, dim1=1, dim2=2)
            all_own.append(torch.diagonal(E, dim1=1, dim2=2)[own_m].cpu())
            eye = torch.eye(n, dtype=torch.bool, device=E.device)
            all_cross.append(E[M & ~eye.unsqueeze(0)].cpu())

    own   = torch.cat(all_own).numpy()
    cross = torch.cat(all_cross).numpy() if all_cross else np.array([])

    rep = {
        "own_frac_negative":  float((own < 0).mean()) if own.size else float("nan"),
        "own_median":         float(np.median(own)) if own.size else float("nan"),
        "own_p10":            float(np.percentile(own, 10)) if own.size else float("nan"),
        "own_p90":            float(np.percentile(own, 90)) if own.size else float("nan"),
        "own_frac_in_m5_0":   float(((own >= -5)  & (own <= 0)).mean()) if own.size else float("nan"),
        "own_frac_in_m10_0":  float(((own >= -10) & (own <= 0)).mean()) if own.size else float("nan"),
    }
    if len(cross) > 0:
        rep.update({
            "cross_median":        float(np.median(cross)),
            "cross_frac_negative": float((cross < 0).mean()),
            "cross_frac_in_m1_1":  float(((cross >= -1) & (cross <= 1)).mean()),
            "cross_abs_median":    float(np.median(np.abs(cross))),
        })
    else:
        rep.update({
            "cross_median": float("nan"),
            "cross_frac_negative": float("nan"),
            "cross_frac_in_m1_1": float("nan"),
            "cross_abs_median": float("nan"),
        })
    return rep

print("Plausibility report defined")

def _empty_loss_acc():
    return dict(fit_num=0.0, fit_den=0.0, sm_num=0.0, sm_den=0.0, el_num=0.0, el_den=0.0)

def _accum_loss(acc, logs):
    n_fit = logs["n_obs"].item()
    n_sm  = logs["n_smooth"].item()
    n_el  = logs["n_elast"].item()
    acc["fit_num"] += logs["loss_fit"].item()    * n_fit
    acc["fit_den"] += n_fit
    acc["sm_num"]  += logs["loss_smooth"].item() * n_sm
    acc["sm_den"]  += n_sm
    acc["el_num"]  += logs["loss_elast"].item()  * n_el
    acc["el_den"]  += n_el

def _epoch_loss(acc, loss_fn):
    return (
        acc["fit_num"] / max(acc["fit_den"], 1.0)
        + loss_fn.lambda_smooth * (acc["sm_num"] / max(acc["sm_den"], 1.0))
        + loss_fn.lambda_elast  * (acc["el_num"] / max(acc["el_den"], 1.0))
    )

print("Metric helpers defined")

Plausibility report defined
Metric helpers defined


# Training Loop

In [ ]:
# Two-phase (warm-start) training loop with AMP, grad clipping and early stopping.
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, neighbor_meta, phase_name="",
                 verbose=False):

    best_val_loss = float("inf")
    no_improve    = 0
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    for epoch in range(n_epochs):
        model.train()
        train_acc = _empty_loss_acc()

        for batch in train_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true   = batch["demands"]
            obs_mask = batch["obs_mask"]

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                    loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                         aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                         aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                         availability=batch["availability"], price_observed=batch["price_observed"])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                     aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                     aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                     availability=batch["availability"], price_observed=batch["price_observed"])
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            _accum_loss(train_acc, logs)

        model.eval()
        val_acc = _empty_loss_acc()
        with torch.no_grad():
            for batch in val_loader:
                batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                y_true   = batch["demands"]
                obs_mask = batch["obs_mask"]

                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                _, logs = loss_fn(y_hat, y_true, obs_mask,
                                  aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                  aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                  price_observed=batch["price_observed"], availability=batch["availability"])

                _accum_loss(val_acc, logs)

        val_loss = _epoch_loss(val_acc, loss_fn)
        prev_lr  = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        new_lr   = optimizer.param_groups[0]["lr"]
        if new_lr < prev_lr:
            no_improve = 0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")

        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping at epoch {epoch+1}")
            break

    return best_val_loss

print("run_training defined")

print("run_training defined")

run_training defined


# Build and Train

In [ ]:
# Builds and trains one model for a given variant / fold, returns its metrics.
def build_and_train(params, variant_cfg, train_fold, val_fold, fold_id, seed, trial_id=0):
    set_all_seeds(seed)

    use_cross     = variant_cfg["use_cross"]
    use_attention = variant_cfg["use_attention"]
    use_splines   = variant_cfg["use_splines"]
    tag           = variant_cfg["name"]

    # ── Economic-constraint knobs (defaults reproduce the original model) ──
    enforce_neg = variant_cfg.get("enforce_negative_beta", True)  # hard sign constraint
    l_own   = variant_cfg.get("l_own",   -5.0)                    # elasticity penalty bounds
    r_own   = variant_cfg.get("r_own",    0.0)
    l_cross = variant_cfg.get("l_cross", -1.0)
    r_cross = variant_cfg.get("r_cross",  1.0)

    # Build the fold dataframes (raw + smoothed) and the DataLoaders.
    train_wide, val_wide, train_wide_s, val_wide_s = build_fold_frames(
        train_wide=train_fold, val_wide=val_fold, smooth_window=SMOOTH_WINDOW,
    )
    train_loader_p0, val_loader_p0, train_loader, val_loader = build_loaders(
        train_wide, val_wide, train_wide_s, val_wide_s, batch_size=params["BATCH_SIZE"]
    )

    # Unpack hyperparameters.
    n_basis       = params["N_BASIS"]
    hidden        = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
    dropout       = params["DROPOUT"]
    act           = params.get("ACT", "gelu")
    lr_p0         = params["LR_P0"]
    lr_p1         = params["LR_P1"]
    lambda_smooth = params["LAMBDA_SMOOTH"]
    lambda_elast  = params["LAMBDA_ELAST"]

    ckpt_p0 = CKPT_DIR / f"{tag}_trial{trial_id}_fold{fold_id}_seed{seed}_p0.pt"
    ckpt_p1 = CKPT_DIR / f"{tag}_trial{trial_id}_fold{fold_id}_seed{seed}_p1.pt"

    # ── Build the spline basis (needed even for no_splines to construct the model) ──
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        obs = train_wide[f"price_observed_{i}"].astype(bool)
        x_i = train_wide.loc[obs, f"log_price_{i}"].values
        if len(x_i) < 10:
            raise ValueError(f"Too few observed prices for UPC {i} to build splines")
        config = builder.build_from_data(
            x_i, n_basis=n_basis, q_min=0.05, q_max=0.95, basis_type="truncated_cubic",
        )
        spline_configs.append(config)
    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"] for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]  for cfg in spline_configs])
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    # ── Context token builder ──
    token_builder = ProductTokenBuilder(
        n=n_upcs,
        n_stores=n_stores, d_store=D_STORE,
        n_brands=n_brands, d_brand=D_BRAND,
        n_styles=n_styles, d_style=D_STYLE,
    )

    # ── make_model threads the variant flags (use_cross + use_attention) ──
    def make_model():
        head = IntegrableDemandHead(
            context_dim=token_builder.d_token,
            K_splines=n_basis,
            n=n_upcs,
            k_neighbors=K_NEIGHBORS,
            hidden=hidden,
            act=act,
            dropout=dropout,
            use_cross=use_cross,
            enforce_negative_beta=enforce_neg,
        )
        model = ICDN(
            context_builder=token_builder,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)
        if use_cross and (not use_attention):
            disable_attention(model)   # uniform edge weights, same graph
        return model

    # ── PHASE 0: log-linear warm start on smoothed targets ─────────────
    m0 = make_model()
    zero_and_freeze_nonlinear(m0)
    init_beta_prior(m0, BETA_EDA, enforce_negative_beta=enforce_neg)
    if use_cross:                      # cross heads only exist when use_cross=True
        with torch.no_grad():
            m0.head.param_head.head_beta_cross.weight.zero_()
            m0.head.param_head.head_beta_cross.bias.zero_()

    loss_p0 = ElasticityLoss(
        huber_delta=1.0, lambda_smooth=lambda_smooth, lambda_elast=lambda_elast,
        l_own=l_own, r_own=r_own, l_cross=l_cross, r_cross=r_cross,  
        reduction="mean",
    )

    decay, no_decay = [], []
    for name, p in m0.named_parameters():
        if not p.requires_grad:
            continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)
    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p0,
    )
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m0, train_loader_p0, val_loader_p0, loss_p0,
                 opt_p0, sch_p0, N_EPOCHS_P0, ES_PATIENCE,
                 ckpt_p0, device, neighbor_meta, "P0")

    # ── PHASE 1: unlock splines on raw targets (unless splines are ablated) ──
    m1 = make_model()
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    if use_splines:
        unfreeze_nonlinear(m1)
    else:
        zero_and_freeze_nonlinear(m1)    # keep splines OFF in phase 1

    loss_p1 = ElasticityLoss(
        huber_delta=1.0, lambda_smooth=lambda_smooth, lambda_elast=lambda_elast,
        l_own=l_own, r_own=r_own, l_cross=l_cross, r_cross=r_cross,  # <── variant bounds
        reduction="mean",
    )
    decay, no_decay = [], []
    for name, p in m1.named_parameters():
        if not p.requires_grad:
            continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)
    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p1,
    )
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m1, train_loader, val_loader, loss_p1,
                 opt_p1, sch_p1, N_EPOCHS_P1, ES_PATIENCE,
                 ckpt_p1, device, neighbor_meta, "P1")
    m1.load_state_dict(torch.load(ckpt_p1, map_location=device))

    # ── Freeze the sparse graph (guarded: selector is None when use_cross=False) ──
    m1.eval()
    selector = m1.head.neighbor_selector
    if selector is not None:
        def h_iter(loader):
            with torch.no_grad():
                for batch in loader:
                    batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                    tokens = m1.context_builder(batch)
                    h      = m1.head.encoder(tokens)
                    yield h
        global_mean = selector.accumulate_mean_scores(
            h_iter(train_loader),
            category=neighbor_meta["category"], brand=neighbor_meta["brand"],
            style=neighbor_meta["style"], liters=neighbor_meta["liters"],
        )
        selector.freeze_graph(
            global_mean,
            category=neighbor_meta["category"], brand=neighbor_meta["brand"],
            style=neighbor_meta["style"], liters=neighbor_meta["liters"],
        )

    # ── Evaluate ──
    pred_metrics  = compute_global_metrics(m1, val_loader, device)
    elast_metrics = compute_elasticity_score(m1, val_loader, device)
    plaus_report  = compute_plausibility_report(m1, val_loader, device)   # <── raw, target-free
    out = {
        "variant": tag, "trial_id": trial_id, "fold": fold_id, "seed": seed,
        "n_train": len(train_wide), "n_val": len(val_wide),
        "enforce_negative_beta": enforce_neg,
        "l_own": l_own, "r_own": r_own, "l_cross": l_cross, "r_cross": r_cross,
        **pred_metrics, **elast_metrics, **plaus_report,
    }

    print(
        f"[{tag}] trial={trial_id} fold={fold_id} | "
        f"R2={out['r2_val']:.4f} MAE={out['mae_val']:.4f} | "
        f"ElastScore={out['elast_score']:.4f} | "
        f"own[pct={100*out['own_in_range']:.1f}% med={out['own_elasticity_median']:.2f}] "
        f"cross[pct={100*out['cross_in_range']:.1f}% med={out['cross_elasticity_median']:.2f}]"
    )

    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)
    return out

print("build_and_train defined")

build_and_train defined


# Variant table

In [ ]:
# Leave-one-out variants. fix_* pins a hyperparameter that is inert for the
# variant, so Optuna does not waste trials searching an irrelevant dimension.
VARIANTS = {
    # ── Family 1: module ablation (referee critique on component contribution) ──
    "full":         dict(use_cross=True,  use_attention=True,  use_splines=True),
    "no_smooth":    dict(use_cross=True,  use_attention=True,  use_splines=True,  fix_lambda_smooth=0.0),
    "no_elast":     dict(use_cross=True,  use_attention=True,  use_splines=True,  fix_lambda_elast=0.0),
    "no_attention": dict(use_cross=True,  use_attention=False, use_splines=True),
    "no_cross":     dict(use_cross=False, use_attention=True,  use_splines=True),
    "no_splines":   dict(use_cross=True,  use_attention=True,  use_splines=False,
                         fix_lambda_smooth=0.0, fix_n_basis=2),

    # ── Family 2: economic-constraint sensitivity (referee critique #4) ──
    # Sign constraint OFF, penalty kept: isolates the hard negativity constraint.
    "free_sign":    dict(use_cross=True,  use_attention=True,  use_splines=True,
                         enforce_negative_beta=False, select_by="robust_r2"),

    # Sign OFF and elasticity penalty OFF: does plausibility EMERGE from data?
    # Selected by predictive score only -> no circularity with the [-5,0]/[-1,1] ranges.
    "unconstrained":dict(use_cross=True,  use_attention=True,  use_splines=True,
                         enforce_negative_beta=False, fix_lambda_elast=0.0,
                         select_by="robust_r2"),

    # Range sensitivity: penalty ON but with much wider, near-inert bounds.
    "wide_bounds":  dict(use_cross=True,  use_attention=True,  use_splines=True,
                         l_own=-10.0, r_own=2.0, l_cross=-3.0, r_cross=3.0,
                         select_by="robust_r2"),
}
for name, cfg in VARIANTS.items():
    cfg["name"] = name

PARAM_COLS = ["N_BASIS", "HIDDEN_KEY", "DROPOUT", "LR_P0", "LR_P1",
              "LAMBDA_SMOOTH", "LAMBDA_ELAST", "BATCH_SIZE"]
print("Variants:", list(VARIANTS.keys()))

Variants: ['full', 'no_smooth', 'no_elast', 'no_attention', 'no_cross', 'no_splines', 'free_sign', 'unconstrained', 'wide_bounds']


# Optuna 

In [ ]:
trial_records = []

def suggest_params(trial, variant_cfg):
    return {
        "N_BASIS": (
            variant_cfg["fix_n_basis"] if "fix_n_basis" in variant_cfg
            else trial.suggest_int("N_BASIS", 2, 16)
        ),
        "HIDDEN_KEY": trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS.keys())),
        "DROPOUT":    trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR_P0":      trial.suggest_float("LR_P0", 1e-4, 1e-2, log=True),
        "LR_P1":      trial.suggest_float("LR_P1", 1e-5, 5e-3, log=True),
        "LAMBDA_SMOOTH": (
            variant_cfg["fix_lambda_smooth"] if "fix_lambda_smooth" in variant_cfg
            else trial.suggest_float("LAMBDA_SMOOTH", 1e-5, 0.2, log=True)
        ),
        "LAMBDA_ELAST": (
            variant_cfg["fix_lambda_elast"] if "fix_lambda_elast" in variant_cfg
            else trial.suggest_float("LAMBDA_ELAST", 1e-5, 0.2, log=True)
        ),
        "BATCH_SIZE": trial.suggest_categorical("BATCH_SIZE", [256, 512, 1024]),
    }


def objective(trial, fold_splits, study_tag: str, variant_cfg: dict):
    params = suggest_params(trial, variant_cfg)
    print(f"\n{'='*70}\n[{study_tag}] Trial {trial.number}")
    for k, v in params.items():
        print(f"  {k}: {v}")
    print("=" * 70)

    run_rows = []
    for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
        for seed in TUNE_SEEDS:
            row = build_and_train(
                params=params, variant_cfg=variant_cfg,
                train_fold=train_fold, val_fold=val_fold,
                fold_id=fold_id, seed=seed, trial_id=trial.number,
            )
            run_rows.append(row)

    df_trial = pd.DataFrame(run_rows)
    mean_r2 = float(df_trial["r2_val"].mean())
    std_r2  = float(df_trial["r2_val"].std(ddof=1)) if len(df_trial) > 1 else 0.0
    mean_elast = float(df_trial["elast_score"].mean())
    std_elast  = float(df_trial["elast_score"].std(ddof=1)) if len(df_trial) > 1 else 0.0
    robust_r2 = mean_r2 - 0.25 * std_r2
    robust_elast = mean_elast - 0.25 * std_elast

    trial.set_user_attr("mean_r2", mean_r2)
    trial.set_user_attr("std_r2", std_r2)
    trial.set_user_attr("mean_elast_score", mean_elast)
    trial.set_user_attr("std_elast_score", std_elast)
    trial.set_user_attr("mean_mae", float(df_trial["mae_val"].mean()))
    trial.set_user_attr("mean_rmse", float(df_trial["rmse_val"].mean()))
    trial.set_user_attr("robust_r2", robust_r2)
    trial.set_user_attr("robust_elast", robust_elast)
    trial.set_user_attr("params_full", params)

    df_trial["trial"] = trial.number
    df_trial["study_tag"] = study_tag
    df_trial["variant"] = variant_cfg["name"]
    for k, v in params.items():
        df_trial[k] = v
    trial_records.extend(df_trial.to_dict(orient="records"))

    print(
        f"[{study_tag}] Trial {trial.number} | "
        f"mean_R2={mean_r2:.4f} robust_R2={robust_r2:.4f} | "
        f"mean_Elast={mean_elast:.4f} robust_Elast={robust_elast:.4f}"
    )
    return robust_r2, robust_elast


def study_to_summary(study) -> pd.DataFrame:
    rows = []
    for t in study.trials:
        if t.values is None:
            continue
        pf = t.user_attrs.get("params_full", t.params)
        rows.append({
            "trial": t.number,
            "mean_r2": t.user_attrs.get("mean_r2", np.nan),
            "std_r2": t.user_attrs.get("std_r2", np.nan),
            "robust_r2": t.user_attrs.get("robust_r2", np.nan),
            "robust_elast": t.user_attrs.get("robust_elast", np.nan),
            "mean_elast_score": t.user_attrs.get("mean_elast_score", np.nan),
            "std_elast_score": t.user_attrs.get("std_elast_score", np.nan),
            "mean_mae": t.user_attrs.get("mean_mae", np.nan),
            "mean_rmse": t.user_attrs.get("mean_rmse", np.nan),
            **pf,
        })
    df = pd.DataFrame(rows)
    df["robust_score"] = df["robust_r2"].fillna(0.0) + df["robust_elast"].fillna(0.0)
    return df.sort_values("robust_score", ascending=False)


def pick_best_row(summary: pd.DataFrame, variant_cfg: dict) -> pd.Series:
    key = variant_cfg.get("select_by", "robust_score")
    return summary.sort_values(key, ascending=False).iloc[0]


def best_payload(best_row: pd.Series, variant_cfg: dict, extra: dict | None = None) -> dict:
    payload = {
        "variant": variant_cfg["name"],
        "protocol": PROTOCOL,
        "trial": int(best_row["trial"]),
        "select_by": variant_cfg.get("select_by", "robust_score"),
        "robust_score": float(best_row["robust_score"]),
        "robust_r2": float(best_row["robust_r2"]),
        "mean_r2": float(best_row["mean_r2"]),
        "std_r2": float(best_row["std_r2"]),
        "mean_elast_score": float(best_row["mean_elast_score"]),
        "std_elast_score": float(best_row["std_elast_score"]),
        "params": {
            "N_BASIS":       int(best_row["N_BASIS"]),
            "HIDDEN_KEY":    str(best_row["HIDDEN_KEY"]),
            "DROPOUT":       float(best_row["DROPOUT"]),
            "LR_P0":         float(best_row["LR_P0"]),
            "LR_P1":         float(best_row["LR_P1"]),
            "LAMBDA_SMOOTH": float(best_row["LAMBDA_SMOOTH"]),
            "LAMBDA_ELAST":  float(best_row["LAMBDA_ELAST"]),
            "BATCH_SIZE":    int(best_row["BATCH_SIZE"]),
        },
    }
    if extra:
        payload.update(extra)
    return payload


def run_optuna(fold_splits, tag: str, n_trials: int, variant_cfg: dict):
    study = optuna.create_study(
        directions=["maximize", "maximize"],
        study_name=tag,
        storage=f"sqlite:///{NESTED_ABL_DIR / (tag + '.db')}",
        load_if_exists=True,
    )
    n_done = len([t for t in study.trials if t.values is not None])
    n_left = max(0, n_trials - n_done)
    print(f"{tag}: {n_done} done, {n_left} left")
    if n_left:
        study.optimize(
            lambda t, splits=fold_splits, tag=tag, cfg=variant_cfg: objective(
                t, splits, tag, cfg
            ),
            n_trials=n_left,
        )
    return study

print("Nested Optuna helpers defined")

# Study

In [ ]:
# Tune: 9 variantes × (5 outer + 1 holdout) × 20 trials × 3 inner × 3 seeds.
# No uses ablation_studies.db (protocolo viejo).
all_outer_best = {}

for name, cfg in VARIANTS.items():
    print(f"\n{'#'*70}\n# VARIANT: {name}\n{'#'*70}")
    outer_best = []
    for plan in nested_plans:
        k = plan["outer_id"]
        tag = f"ablation_{name}_outer{k}"
        study = run_optuna(plan["inner_splits"], tag, N_TRIALS_NESTED, cfg)
        summary = study_to_summary(study)
        summary["outer_id"] = k
        summary["variant"] = name
        best_row = pick_best_row(summary, cfg)
        payload = best_payload(best_row, cfg, extra={"outer_id": k})
        path_json = NESTED_ABL_DIR / f"{name}_outer{k}_best_params.json"
        path_csv  = NESTED_ABL_DIR / f"{name}_outer{k}_trials.csv"
        with open(path_json, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        summary.to_csv(path_csv, index=False)
        outer_best.append(payload)
        print(f"Saved {tag} trial {payload['trial']} robust_score={payload['robust_score']:.4f}")

    with open(NESTED_ABL_DIR / f"{name}_outer_best_params.json", "w", encoding="utf-8") as f:
        json.dump(outer_best, f, indent=2, ensure_ascii=False)
    all_outer_best[name] = outer_best

    tag_h = f"ablation_{name}_holdout"
    study_h = run_optuna(holdout_inner, tag_h, N_TRIALS_HOLDOUT, cfg)
    summary_h = study_to_summary(study_h)
    best_h = pick_best_row(summary_h, cfg)
    payload_h = best_payload(best_h, cfg, extra={"split": "holdout"})
    with open(NESTED_ABL_DIR / f"{name}_holdout_best_params.json", "w", encoding="utf-8") as f:
        json.dump(payload_h, f, indent=2, ensure_ascii=False)
    summary_h.to_csv(NESTED_ABL_DIR / f"{name}_holdout_trials.csv", index=False)
    print(f"Saved {tag_h} trial {payload_h['trial']}")

if trial_records:
    pd.DataFrame(trial_records).to_csv(
        NESTED_ABL_DIR / "ablation_tune_records_nested.csv", index=False
    )
print("All nested ablation studies completed")

In [ ]:
eval_rows = []

for name, cfg in VARIANTS.items():
    with open(NESTED_ABL_DIR / f"{name}_outer_best_params.json", encoding="utf-8") as f:
        outer_best = json.load(f)
    params_by_outer = {int(p["outer_id"]): p for p in outer_best}

    for plan in nested_plans:
        k = plan["outer_id"]
        params = params_by_outer[k]["params"]
        for seed in EVAL_SEEDS:
            row = build_and_train(
                params=params, variant_cfg=cfg,
                train_fold=plan["outer_train"], val_fold=plan["outer_val"],
                fold_id=k, seed=seed, trial_id=-1,
            )
            row.update({"variant": name, "outer_id": k, "run_type": "nested_outer_eval"})
            eval_rows.append(row)

    with open(NESTED_ABL_DIR / f"{name}_holdout_best_params.json", encoding="utf-8") as f:
        holdout_best = json.load(f)
    for seed in EVAL_SEEDS:
        row = build_and_train(
            params=holdout_best["params"], variant_cfg=cfg,
            train_fold=train_final, val_fold=val_final,
            fold_id=-1, seed=seed, trial_id=-1,
        )
        row.update({"variant": name, "outer_id": -1, "run_type": "nested_holdout_eval"})
        eval_rows.append(row)

ablation_eval = pd.DataFrame(eval_rows)
ablation_eval.to_csv(NESTED_ABL_DIR / "ablation_eval_nested.csv", index=False)

outer = ablation_eval[ablation_eval["run_type"] == "nested_outer_eval"]
summary = (
    outer.groupby("variant")
    .agg(
        r2_mean=("r2_val", "mean"), r2_std=("r2_val", "std"),
        mae_mean=("mae_val", "mean"),
        elast_mean=("elast_score", "mean"), elast_std=("elast_score", "std"),
        n=("r2_val", "size"),
    )
    .reset_index()
)
display(summary)